<a href="https://colab.research.google.com/github/natasya783/Riwayat-Servis/blob/main/Riwayat%20Servis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
3. RIWAYAT SERVIS (Bobot 15%)
# =====================================================================
def tambah_riwayat_servis(user_id):
    """Mencatat transaksi servis dan menyimpan target servis berikutnya."""
    print("\n=== TAMBAH RIWAYAT SERVIS ===")
    kendaraan = pilih_kendaraan(user_id)
    if not kendaraan:
        return

    print("Jenis servis:")
    for nomor, jenis in enumerate(INTERVAL_SERVIS, start=1):
        print(f"{nomor}. {jenis} ({INTERVAL_SERVIS[jenis]:,} Km)")
    pilihan = input_angka("Pilih jenis servis: ", minimum=1)
    daftar_jenis = list(INTERVAL_SERVIS.keys())
    if pilihan > len(daftar_jenis):
        print("ERROR: Pilihan jenis servis tidak tersedia.")
        return

    jenis = daftar_jenis[pilihan - 1]
    tanggal = input("Tanggal servis (DD-MM-YYYY): ").strip()
    try:
        datetime.strptime(tanggal, "%d-%m-%Y")
    except ValueError:
        print("ERROR: Format tanggal harus DD-MM-YYYY.")
        return

    km_servis = input_angka("KM saat servis: ", minimum=0)
    biaya = input_angka("Biaya servis (Rp): ", minimum=0)
    catatan = input("Catatan (boleh dikosongkan): ").strip()

    if km_servis > kendaraan[5]:
        print("ERROR: KM saat servis tidak boleh melebihi KM kendaraan saat ini.")
        return

    # Memanggil fungsi perhitungan target servis
    target = hitung_target_servis(jenis, km_servis)

    with koneksi_db() as conn:
        conn.execute("""
            INSERT INTO riwayat_servis
            (id_kendaraan, jenis_servis, km_saat_servis, tanggal_servis,
             biaya, km_target_berikutnya, catatan)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (kendaraan[0], jenis, km_servis, tanggal, biaya, target, catatan))
        conn.commit()
    print(f"Riwayat servis tersimpan. Target {jenis} berikutnya: {target:,} Km.")

def tampilkan_riwayat(user_id):
    """Menampilkan seluruh riwayat servis kendaraan yang dipilih."""
    kendaraan = pilih_kendaraan(user_id)
    if not kendaraan:
        return
    with koneksi_db() as conn:
        cur = conn.execute("""
            SELECT tanggal_servis, jenis_servis, km_saat_servis, biaya, catatan
            FROM riwayat_servis WHERE id_kendaraan = ?
            ORDER BY id
        """, (kendaraan[0],))
        data = cur.fetchall()

    print("\n=== RIWAYAT SERVIS ===")
    if not data:
        print("Belum ada riwayat servis.")
        return
    print(f"{'TANGGAL':<13} {'JENIS SERVIS':<16} {'KM':>9} {'BIAYA (Rp)':>13}  CATATAN")
    for tanggal, jenis, km, biaya, catatan in data:
        print(f"{tanggal:<13} {jenis:<16} {km:>9,} {biaya:>13,}  {catatan or '-'}")